In [9]:
# Celda 1 - Librerias  [V12 - Anti-Ghost + GP Depth]
import cv2
import numpy as np
import time
import math
import threading
import collections
import customtkinter as ctk
from PIL import Image
from ultralytics import YOLO
import os
import operator

# Librerias para Optimizador DEAP (GP Profundidad)
import random
from deap import base, creator, tools, algorithms, gp

ctk.set_appearance_mode('dark')
print('Celda 1 V12: lista.')


Celda 1 V12: lista.


In [10]:
# Celda 2 - Motor de Vision V12 [Anti-Ghost + Blur + GP Depth + Circulo]
from pygrabber.dshow_graph import FilterGraph
import cv2
import numpy as np
import time
import math
import collections

def detectar_camaras_sistema():
    try:
        graph = FilterGraph()
        return [(i, n) for i, n in enumerate(graph.get_input_devices())]
    except Exception as e:
        print(f'Error al buscar camaras: {e}')
        return []

LIMITES_PELOTAS = {'Rojo': 10, 'Negro': 10, 'Blanco': 10}
MAX_PELOTAS_TOTAL = 10

# Radio del circulo de deteccion como fraccion del lado menor
MESH_FRACTION = 0.38

# ─────────────────────────────────────────────────────────────
# 1. DETECCION DE MOVIMIENTO BRUSCO (BLUR)
# ─────────────────────────────────────────────────────────────
class DetectorMovimiento:
    """
    Detecta frames borrosos (movimiento brusco de camara) usando
    la varianza del Laplaciano. Si la varianza cae por debajo
    del umbral, el frame se considera borroso y se salta la
    deteccion para evitar fantasmas.
    
    Tambien detecta movimiento excesivo entre frames consecutivos
    comparando diferencia absoluta promedio.
    """
    def __init__(self, umbral_blur=45.0, umbral_movimiento=25.0, 
                 ventana_estabilidad=3):
        self.umbral_blur        = umbral_blur
        self.umbral_movimiento  = umbral_movimiento
        self.ventana_estabilidad = ventana_estabilidad
        self.prev_gray          = None
        self.frames_estables    = 0
    
    def frame_valido(self, frame):
        """Retorna True si el frame es estable y nitido."""
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
        # Test 1: Nitidez (Laplaciano)
        laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()
        nitido = laplacian_var >= self.umbral_blur
        
        # Test 2: Movimiento entre frames consecutivos
        movimiento_ok = True
        if self.prev_gray is not None:
            diff = cv2.absdiff(gray, self.prev_gray)
            mov_promedio = diff.mean()
            movimiento_ok = mov_promedio < self.umbral_movimiento
        self.prev_gray = gray.copy()
        
        # Requiere N frames estables consecutivos antes de detectar
        if nitido and movimiento_ok:
            self.frames_estables = min(self.frames_estables + 1, 
                                       self.ventana_estabilidad + 5)
        else:
            self.frames_estables = 0
        
        return self.frames_estables >= self.ventana_estabilidad

# ─────────────────────────────────────────────────────────────
# 2. NMS INTRA-COLOR (evita duplicados del mismo color)
# ─────────────────────────────────────────────────────────────
def nms_mismo_color(detecciones, factor_radio=1.2):
    """
    Si dos detecciones del MISMO color tienen centros mas cercanos
    que factor_radio * (r1 + r2), son la misma pelota.
    Conserva la de mayor radio (mas confiable).
    """
    if len(detecciones) <= 1:
        return detecciones
    # Ordenar por radio descendente (mas grande primero)
    ordenadas = sorted(detecciones, key=lambda d: d[2], reverse=True)
    resultado = []
    for d in ordenadas:
        cx, cy, r = d
        es_duplicado = False
        for f in resultado:
            fx, fy, fr = f
            dist = math.hypot(cx - fx, cy - fy)
            umbral = (r + fr) * factor_radio
            if dist < umbral:
                es_duplicado = True
                break
        if not es_duplicado:
            resultado.append(d)
    return resultado

# ─────────────────────────────────────────────────────────────
# 3. GP PARA ESTIMACION DE PROFUNDIDAD (DEAP)
# ─────────────────────────────────────────────────────────────
def _div_protegida(a, b):
    return a / b if abs(b) > 1e-6 else a

def _sqrt_protegida(a):
    return math.sqrt(abs(a))

def _log_protegido(a):
    return math.log(abs(a) + 1e-6)

def evolucionar_estimador_profundidad():
    """
    Usa Programacion Genetica (DEAP) para evolucionar una formula
    optima de estimacion de profundidad z = f(radio_px, frame_h, frame_w).
    
    Datos de calibracion sinteticos basados en modelo pinhole:
        z = (f_focal * D_real) / (2 * radio_px)
    
    Para una pelota de ~6.5cm de diametro, camara 640x480, FOV ~60 grados:
        f_focal ≈ fw / (2 * tan(FOV/2)) ≈ 554 px
        
    Distancia real -> radio esperado en pixels:
        0.3m -> ~60px,  0.5m -> ~36px,  1.0m -> ~18px,
        1.5m -> ~12px,  2.0m -> ~9px,   3.0m -> ~6px
    """
    # Datos de calibracion: (radio_px, frame_h, frame_w, z_real)
    datos = [
        (80,  480, 640, 0.22),
        (60,  480, 640, 0.30),
        (45,  480, 640, 0.40),
        (36,  480, 640, 0.50),
        (27,  480, 640, 0.67),
        (18,  480, 640, 1.00),
        (14,  480, 640, 1.29),
        (12,  480, 640, 1.50),
        (9,   480, 640, 2.00),
        (7,   480, 640, 2.57),
        (6,   480, 640, 3.00),
        (4,   480, 640, 4.50),
        # Resolucion 1280x720
        (120, 720, 1280, 0.30),
        (72,  720, 1280, 0.50),
        (36,  720, 1280, 1.00),
        (18,  720, 1280, 2.00),
        (12,  720, 1280, 3.00),
    ]
    
    # Configurar GP
    if hasattr(creator, 'FitnessMinDepth'):
        del creator.FitnessMinDepth
    if hasattr(creator, 'IndividualDepth'):
        del creator.IndividualDepth
    
    creator.create('FitnessMinDepth', base.Fitness, weights=(-1.0,))
    creator.create('IndividualDepth', gp.PrimitiveTree, fitness=creator.FitnessMinDepth)
    
    pset = gp.PrimitiveSet('DEPTH', 3)  # radio, fh, fw
    pset.renameArguments(ARG0='r', ARG1='fh', ARG2='fw')
    
    pset.addPrimitive(operator.add, 2)
    pset.addPrimitive(operator.sub, 2)
    pset.addPrimitive(operator.mul, 2)
    pset.addPrimitive(_div_protegida, 2)
    pset.addPrimitive(_sqrt_protegida, 1)
    pset.addPrimitive(_log_protegido, 1)
    pset.addPrimitive(operator.neg, 1)
    pset.addPrimitive(abs, 1)
    
    for c in [0.01, 0.065, 0.1, 0.5, 1.0, 2.0, 3.14, 10.0, 100.0, 554.0]:
        pset.addTerminal(c)
    
    toolbox_gp = base.Toolbox()
    toolbox_gp.register('expr', gp.genHalfAndHalf, pset=pset, min_=2, max_=5)
    toolbox_gp.register('individual', tools.initIterate, creator.IndividualDepth, toolbox_gp.expr)
    toolbox_gp.register('population', tools.initRepeat, list, toolbox_gp.individual)
    toolbox_gp.register('compile', gp.compile, pset=pset)
    
    def evaluar(individuo):
        func = toolbox_gp.compile(expr=individuo)
        error_total = 0.0
        for r_px, fh, fw, z_real in datos:
            try:
                z_pred = func(float(r_px), float(fh), float(fw))
                z_pred = max(0.05, min(10.0, float(z_pred)))
            except:
                return (1e6,)
            error_total += (z_pred - z_real) ** 2
        return (error_total / len(datos),)
    
    toolbox_gp.register('evaluate', evaluar)
    toolbox_gp.register('select', tools.selTournament, tournsize=4)
    toolbox_gp.register('mate', gp.cxOnePoint)
    toolbox_gp.register('expr_mut', gp.genFull, min_=1, max_=3)
    toolbox_gp.register('mutate', gp.mutUniform, expr=toolbox_gp.expr_mut, pset=pset)
    
    # Limitar profundidad del arbol para evitar bloat
    toolbox_gp.decorate('mate', gp.staticLimit(key=operator.attrgetter('height'), max_value=8))
    toolbox_gp.decorate('mutate', gp.staticLimit(key=operator.attrgetter('height'), max_value=8))
    
    random.seed(42)
    pop = toolbox_gp.population(n=300)
    hof = tools.HallOfFame(1)
    
    algorithms.eaSimple(pop, toolbox_gp,
                         cxpb=0.6, mutpb=0.3, ngen=50,
                         halloffame=hof, verbose=False)
    
    mejor = hof[0]
    func_mejor = toolbox_gp.compile(expr=mejor)
    fitness_mejor = evaluar(mejor)[0]
    print(f'GP Depth: MSE={fitness_mejor:.4f}, expr={str(mejor)[:80]}')
    return func_mejor

# Ejecutar GP al cargar la celda
print('Evolucionando estimador de profundidad con GP...')
_GP_DEPTH_FUNC = evolucionar_estimador_profundidad()

def estimar_z_gp(radio_px, frame_shape):
    """Usa la funcion evolucionada por GP para estimar Z."""
    fh, fw = frame_shape[:2]
    try:
        z = _GP_DEPTH_FUNC(float(radio_px), float(fh), float(fw))
        return round(max(0.1, min(5.0, float(z))), 2)
    except:
        # Fallback simple
        frac = (math.pi * radio_px * radio_px) / (fh * fw)
        return round(max(0.1, min(5.0, 1.0 / (frac * 10 + 0.01))), 2)


# ─────────────────────────────────────────────────────────────
# 4. RANGOS HSV / COLOR
# ─────────────────────────────────────────────────────────────
HSV_RANGOS = {
    'Rojo': [
        (np.array([0,   70, 50]),  np.array([12,  255, 255])),
        (np.array([160, 70, 50]),  np.array([180, 255, 255])),
    ],
    'Blanco': [
        (np.array([0,  0, 150]),   np.array([180, 55,  255])),
    ],
    'Negro': [
        (np.array([0,  0,   0]),   np.array([180, 255,  75])),
    ],
}

def get_color_bgr(nombre):
    n = nombre.lower()
    if 'rojo'   in n: return (0,   0, 220)
    if 'blanco' in n: return (200, 200, 200)
    if 'negro'  in n: return (80,  80,  80)
    return (0, 220, 220)

# ─────────────────────────────────────────────────────────────
# 5. ZONA DE DETECCION (CIRCULO) + RETICULA
# ─────────────────────────────────────────────────────────────
def get_zona_deteccion(frame_shape):
    fh, fw = frame_shape[:2]
    cx, cy = fw // 2, fh // 2
    radio = int(min(fw, fh) * MESH_FRACTION)
    return cx, cy, radio

def dibujar_reticula(frame):
    """Dibuja la reticula SIEMPRE (cuadrado + circulo + ejes)."""
    cx, cy, r = get_zona_deteccion(frame.shape)
    c = (255, 255, 255)
    cv2.rectangle(frame, (cx-r, cy-r), (cx+r, cy+r), c, 1)
    cv2.line(frame, (cx, cy-r), (cx, cy+r), c, 1)
    cv2.line(frame, (cx-r, cy), (cx+r, cy), c, 1)
    cv2.circle(frame, (cx, cy), r, c, 1)
    return cx, cy, r

def dentro_del_circulo(px, py, cx, cy, radio):
    return math.hypot(px - cx, py - cy) <= radio

# ─────────────────────────────────────────────────────────────
# 6. VALIDACION DE COLOR HSV
# ─────────────────────────────────────────────────────────────
def validar_color_en_roi(frame, x1, y1, x2, y2, nombre, umbral_frac=0.07):
    fh, fw = frame.shape[:2]
    x1c, y1c = max(0, int(x1)), max(0, int(y1))
    x2c, y2c = min(fw, int(x2)), min(fh, int(y2))
    roi = frame[y1c:y2c, x1c:x2c]
    if roi.size == 0: return False
    rh, rw = roi.shape[:2]
    mask = np.zeros((rh, rw), dtype=np.uint8)
    cv2.circle(mask, (rw//2, rh//2), min(rw, rh)//2, 255, -1)
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    color_mask = np.zeros((rh, rw), dtype=np.uint8)
    for lo, hi in HSV_RANGOS.get(nombre, []):
        color_mask |= cv2.inRange(hsv, lo, hi)
    color_mask &= mask
    total_px = cv2.countNonZero(mask)
    color_px = cv2.countNonZero(color_mask)
    if total_px == 0: return False
    return (color_px / total_px) >= umbral_frac

# ─────────────────────────────────────────────────────────────
# 7. VALIDACION DE FORMA DE PELOTA
# ─────────────────────────────────────────────────────────────
def es_forma_pelota(x1, y1, x2, y2, frame_shape,
                    max_aspect=1.55, min_frac=0.0003, max_frac=0.50):
    w, h = x2 - x1, y2 - y1
    if w <= 0 or h <= 0: return False
    if max(w, h) / (min(w, h) + 1e-9) > max_aspect: return False
    fh, fw = frame_shape[:2]
    frac = (w * h) / (fw * fh)
    return min_frac <= frac <= max_frac

# ─────────────────────────────────────────────────────────────
# 8. TRACKER MEJORADO (Anti-Ghost)
# ─────────────────────────────────────────────────────────────
class TrackerCirculo:
    """
    Tracker con EMA (suavizado exponencial) y confirmacion estricta.
    - frames_conf=4: necesita 4 frames consecutivos para confirmar
    - frames_perdida=3: pierde la pelota rapido si desaparece
    - alpha=0.5: suaviza posicion para evitar saltos bruscos
    """
    def __init__(self, alpha=0.5, frames_conf=4, frames_perdida=3):
        self.alpha          = alpha
        self.frames_conf    = frames_conf
        self.frames_perdida = frames_perdida
        self.reiniciar()

    def reiniciar(self):
        self.suave       = None
        self.conteo_det  = 0
        self.conteo_perd = 0
        self.visible     = False

    def actualizar(self, deteccion):
        if deteccion is None:
            self.conteo_det  = 0
            self.conteo_perd = min(self.conteo_perd + 1, self.frames_perdida + 1)
            if self.conteo_perd >= self.frames_perdida:
                self.visible = False
                self.suave   = None
            return tuple(int(round(v)) for v in self.suave) if self.visible else None

        self.conteo_perd = 0
        self.conteo_det  = min(self.conteo_det + 1, self.frames_conf + 10)

        if self.conteo_det < self.frames_conf:
            # Aun acumulando confirmacion, guardar posicion provisional
            if self.suave is None:
                self.suave = tuple(float(v) for v in deteccion[:3])
            return None

        if self.suave is None:
            self.suave   = tuple(float(v) for v in deteccion[:3])
            self.visible = True
            return tuple(int(round(v)) for v in deteccion[:3])

        # Suavizado EMA para evitar saltos bruscos
        sx = self.alpha * deteccion[0] + (1.0 - self.alpha) * self.suave[0]
        sy = self.alpha * deteccion[1] + (1.0 - self.alpha) * self.suave[1]
        sr = self.alpha * deteccion[2] + (1.0 - self.alpha) * self.suave[2]
        self.suave   = (sx, sy, sr)
        self.visible = True
        return (int(round(sx)), int(round(sy)), int(round(sr)))

def emparejar_detecciones(trackers, detecciones):
    asignaciones = [None] * len(trackers)
    if not detecciones: return asignaciones
    det_usadas = set()
    for i, tr in enumerate(trackers):
        if tr.suave is not None and tr.visible:
            mejor_det  = None
            mejor_dist = float('inf')
            for j, d in enumerate(detecciones):
                if j in det_usadas: continue
                dist = math.hypot(tr.suave[0] - d[0], tr.suave[1] - d[1])
                if dist < mejor_dist and dist < 100:
                    mejor_dist = dist
                    mejor_det  = j
            if mejor_det is not None:
                asignaciones[i] = detecciones[mejor_det]
                det_usadas.add(mejor_det)
    for j, d in enumerate(detecciones):
        if j not in det_usadas:
            for i, tr in enumerate(trackers):
                if asignaciones[i] is None and (tr.suave is None or not tr.visible):
                    asignaciones[i] = d
                    break
    return asignaciones


# ─────────────────────────────────────────────────────────────
# 9. DETECTOR DE MOVIMIENTO GLOBAL (instancia unica)
# ─────────────────────────────────────────────────────────────
_detector_movimiento = DetectorMovimiento(
    umbral_blur=45.0,
    umbral_movimiento=25.0,
    ventana_estabilidad=3
)


# ─────────────────────────────────────────────────────────────
# 10. FUNCION PRINCIPAL DE PROCESAMIENTO
# ─────────────────────────────────────────────────────────────
def procesar_frame_vision_yolo(frame, temporizadores, trackers, modelo_yolo,
                                callback_objeto=None, **kwargs):
    t_act       = time.time()
    hay_objetos = False
    fh, fw      = frame.shape[:2]

    cx_scr, cy_scr, radio_zona = get_zona_deteccion(frame.shape)
    detecciones_brutas = {'Rojo': [], 'Blanco': [], 'Negro': []}

    # ── ANTI-GHOST: Si el frame es borroso o hay movimiento brusco,
    #    NO se ejecuta YOLO. Los trackers reciben None y se limpian solos.
    frame_estable = _detector_movimiento.frame_valido(frame)

    if frame_estable:
        # ── YOLO inference ───────────────────────────────────
        resultados = modelo_yolo.predict(frame, conf=0.55, verbose=False)

        if len(resultados) > 0:
            cajas = resultados[0].boxes
            if cajas is not None:
                for i in range(len(cajas)):
                    cls_id          = int(cajas.cls[i].item())
                    x1, y1, x2, y2 = cajas.xyxy[i].tolist()

                    nombre = None
                    if cls_id == 0: nombre = 'Rojo'
                    elif cls_id == 1: nombre = 'Blanco'
                    elif cls_id == 2: nombre = 'Negro'
                    if nombre is None: continue

                    cx = int((x1 + x2) / 2)
                    cy = int((y1 + y2) / 2)

                    # FILTRO ESPACIAL: centro debe estar DENTRO del circulo
                    if not dentro_del_circulo(cx, cy, cx_scr, cy_scr, radio_zona):
                        continue

                    if not es_forma_pelota(x1, y1, x2, y2, frame.shape):
                        continue

                    if not validar_color_en_roi(frame, x1, y1, x2, y2, nombre):
                        continue

                    w     = x2 - x1
                    h     = y2 - y1
                    radio = int((w + h) / 4)
                    detecciones_brutas[nombre].append((cx, cy, radio))

        # ── NMS intra-color (eliminar duplicados mismo color) ─
        for nombre in detecciones_brutas:
            detecciones_brutas[nombre] = nms_mismo_color(
                detecciones_brutas[nombre], factor_radio=1.2
            )
    else:
        # Frame inestable: indicador visual
        cv2.putText(frame, "ESTABILIZANDO...", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 200, 255), 2)

    # ── Tracking y dibujado ──────────────────────────────────
    total_pelotas = 0
    for nombre in ['Rojo', 'Blanco', 'Negro']:
        lista = sorted(detecciones_brutas[nombre],
                       key=lambda d: math.hypot(d[0]-cx_scr, d[1]-cy_scr))
        if total_pelotas >= MAX_PELOTAS_TOTAL:
            lista = []
        else:
            lista = lista[:MAX_PELOTAS_TOTAL - total_pelotas]

        asignaciones = emparejar_detecciones(trackers[nombre], lista)

        for idx, det in enumerate(asignaciones):
            tr     = trackers[nombre][idx]
            result = tr.actualizar(det)
            if result is None:
                temporizadores[nombre][idx] = 0.0
                continue
            if total_pelotas >= MAX_PELOTAS_TOTAL: break

            hay_objetos    = True
            total_pelotas += 1
            cx, cy, r      = result
            if temporizadores[nombre][idx] == 0.0:
                temporizadores[nombre][idx] = t_act

            color_bgr = get_color_bgr(nombre)
            x_norm    = round(cx / fw * 2 - 1, 2)
            y_norm    = round(1 - cy / fh * 2, 2)
            z_est     = estimar_z_gp(r, frame.shape)

            cv2.circle(frame, (cx, cy), r, color_bgr, 3)
            cv2.circle(frame, (cx, cy), 4, color_bgr, -1)

            lbl   = f'{nombre} #{idx+1}'
            lbl_c = f'X:{x_norm:+.1f} Y:{y_norm:+.1f} Z:{z_est}m'

            label_y     = max(18, cy - r - 10)
            (tw, th), _ = cv2.getTextSize(lbl, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
            cv2.rectangle(frame, (cx-r, label_y-th-4), (cx-r+tw+4, label_y+4), (20,20,20), -1)
            cv2.putText(frame, lbl, (cx-r+2, label_y), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color_bgr, 2)

            cy2         = label_y + th + 6
            (cw, ch), _ = cv2.getTextSize(lbl_c, cv2.FONT_HERSHEY_SIMPLEX, 0.42, 1)
            cv2.rectangle(frame, (cx-r, cy2-ch-2), (cx-r+cw+4, cy2+2), (20,20,20), -1)
            cv2.putText(frame, lbl_c, (cx-r+2, cy2), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (210,210,210), 1)

            if callback_objeto is not None:
                callback_objeto(lbl, x_norm, y_norm, z_est)

    return frame, hay_objetos

print('Celda 2 V12 lista (Anti-Ghost + GP Depth + Circulo + NMS).')


Evolucionando estimador de profundidad con GP...
GP Depth: MSE=0.0611, expr=_sqrt_protegida(sub(mul(mul(_sqrt_protegida(_div_protegida(10.0, r)), _log_prote
Celda 2 V12 lista (Anti-Ghost + GP Depth + Circulo + NMS).


In [11]:
# Celda 3 - UI y Optimizacion OpenCV DEAP ABSOLUTO  [V0.6]
import customtkinter as ctk
from PIL import Image
import threading

class EscanerApp(ctk.CTk):
    def __init__(self):
        super().__init__()
        self.title('Deteccion de Carga - V12 (Anti-Ghost)')
        self.geometry('1380x860')
        self.configure(fg_color='#1a1d2e')
        
        self.cap = None
        self.escaneando = False
        
        self.trackers = {
            'Rojo': [TrackerCirculo() for _ in range(LIMITES_PELOTAS['Rojo'])],
            'Blanco': [TrackerCirculo() for _ in range(LIMITES_PELOTAS['Blanco'])],
            'Negro': [TrackerCirculo() for _ in range(LIMITES_PELOTAS['Negro'])]
        }
        self.temporizadores = {
            'Rojo': [0.0]*LIMITES_PELOTAS['Rojo'],
            'Blanco': [0.0]*LIMITES_PELOTAS['Blanco'],
            'Negro': [0.0]*LIMITES_PELOTAS['Negro']
        }
        self.historial_objetos = set()
        self.puntos_actuales = 0
        self.limite_alcanzado = False
        
        self.cam_delay_ms = 33

        ruta_modelo = r"C:\Users\jrhe0\.gemini\antigravity-ide\scratch\detector_pelotas\models\entrenamiento_pelotas\weights\best.pt"
        if os.path.exists(ruta_modelo):
            print("Cargando modelo de IA (YOLOv8)...")
            self.modelo_yolo = YOLO(ruta_modelo)
        else:
            print("ERROR: No se encontro el modelo YOLO en " + ruta_modelo)
            self.modelo_yolo = None
        self._construir_ui()
        self._auto_check_camaras()

    def _construir_ui(self):
        self.lbl_titulo = ctk.CTkLabel(self, text='CONFIGURACION DE CAMARA', font=('Helvetica', 22, 'bold'), text_color='#e8eaf0')
        self.lbl_titulo.pack(pady=(28, 12))
        self.panel_config = ctk.CTkFrame(self, fg_color='transparent')
        self.panel_config.pack(expand=True, fill='both')
        
        self.btn_detectar = ctk.CTkButton(self.panel_config, text='DETECTAR CAMARAS', command=self._accion_buscar)
        self.btn_detectar.pack(pady=10)
        self.frame_lista = ctk.CTkScrollableFrame(self.panel_config, width=700, height=150)
        self.frame_lista.pack(pady=5)
        self.indice_sel = ctk.StringVar(value='-1')
        self.btn_iniciar = ctk.CTkButton(self.panel_config, text='INICIAR DETECCION', state='disabled', command=self._iniciar_camara)
        self.btn_iniciar.pack(pady=40)
        
        self.panel_video = ctk.CTkFrame(self, fg_color='transparent')
        self.lbl_video = ctk.CTkLabel(self.panel_video, text='')
        self.lbl_video.pack(pady=10, padx=10)
        
        self.frame_controles = ctk.CTkFrame(self.panel_video, fg_color='transparent')
        self.frame_controles.pack(pady=8)
        
        self.btn_escaneo = ctk.CTkButton(self.frame_controles, text='Iniciar Escaneo', command=self._toggle_escaneo)
        self.btn_escaneo.grid(row=0, column=0, padx=10)
        
        self.btn_limpiar = ctk.CTkButton(self.frame_controles, text='Limpiar Todo', command=self._limpiar_todo)
        self.btn_limpiar.grid(row=0, column=1, padx=10)
        
        self.btn_detener = ctk.CTkButton(self.frame_controles, text='Dejar de Escanear', fg_color='#e74c3c', hover_color='#c0392b', command=self._detener_camara)
        self.btn_detener.grid(row=0, column=2, padx=10)
        
        self.lbl_puntos = ctk.CTkLabel(self.frame_controles, text='Puntos: 0 / 10 MAX', font=('Helvetica', 16, 'bold'), text_color='#3498db')
        self.lbl_puntos.grid(row=0, column=3, padx=20)
        
        self.lbl_gp_status = ctk.CTkLabel(self.panel_video, text='', text_color='#f1c40f', font=('Helvetica', 14, 'bold'))
        self.lbl_gp_status.pack(pady=5)

        self.lista_scroll = ctk.CTkScrollableFrame(self.panel_video, width=300)
        self.lista_scroll.pack(side='right', fill='y', padx=10, pady=10)

    def _auto_check_camaras(self):
        if self.cap is not None and self.cap.isOpened():
            self.after(2000, self._auto_check_camaras)
            return
            
        camaras = detectar_camaras_sistema()
        nombres_camaras = [n for _, n in camaras]
        
        if not hasattr(self, 'ultimas_camaras'):
            self.ultimas_camaras = []
            
        if nombres_camaras != self.ultimas_camaras:
            self.ultimas_camaras = nombres_camaras
            self._accion_buscar(camaras)
            
        self.after(2000, self._auto_check_camaras)

    def _accion_buscar(self, camaras=None):
        if camaras is None:
            camaras = detectar_camaras_sistema()
            self.ultimas_camaras = [n for _, n in camaras]
            
        for w in self.frame_lista.winfo_children(): w.destroy()
        if camaras:
            current_sel = self.indice_sel.get()
            for idx, etq in camaras:
                ctk.CTkRadioButton(self.frame_lista, text=etq, variable=self.indice_sel, value=str(idx)).pack(anchor='w', pady=5)
            if not any(str(idx) == current_sel for idx, _ in camaras):
                self.indice_sel.set(str(camaras[0][0]))
            self.btn_iniciar.configure(state='normal')
        else:
            self.btn_iniciar.configure(state='disabled')
            self.indice_sel.set('-1')

    def _iniciar_camara(self):
        idx = int(self.indice_sel.get())
        if idx == -1: return
        self.panel_config.pack_forget()
        self.panel_video.pack(expand=True, fill='both')
        self.lbl_titulo.configure(text='MONITOR EN VIVO (V12)')
        
        self.cap = cv2.VideoCapture(idx, cv2.CAP_DSHOW)
        if not self.cap.isOpened() or not self.cap.read()[0]:
            self.cap = cv2.VideoCapture(idx)
            
        if not self.cap.isOpened():
            self.lbl_video.configure(text='ERROR', text_color='red')
            return
            
        self.lbl_video.configure(text='')
        self.escaneando = False
        self._loop_video()

    def _detener_camara(self):
        self.escaneando = False
        if self.cap: self.cap.release()
        self.panel_video.pack_forget()
        self.panel_config.pack(expand=True, fill='both')

    def _toggle_escaneo(self):
        self.escaneando = not self.escaneando
        if self.escaneando: self.btn_escaneo.configure(text='Pausar Escaneo', fg_color='#d35400')
        else: self.btn_escaneo.configure(text='Reanudar Escaneo', fg_color='#f39c12')

    def _loop_video(self):
        if not self.cap or not self.cap.isOpened(): return
        
        ret, frame = self.cap.read()
        if ret:
            frame = cv2.flip(frame, 1)
            # ── RETICULA SIEMPRE VISIBLE ──────────────────────
            dibujar_reticula(frame)
            # ── Deteccion solo cuando escaneo esta activo ─────
            if self.escaneando and not self.limite_alcanzado:
                if self.modelo_yolo is not None:
                    frame, _ = procesar_frame_vision_yolo(
                        frame, self.temporizadores, self.trackers, self.modelo_yolo,
                        callback_objeto=self.on_objeto_detectado
                    )
                else:
                    cv2.putText(frame, "ERROR: YOLO NO CARGADO", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
            img = ctk.CTkImage(light_image=Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)), size=(920, 630))
            self.lbl_video.configure(image=img)
        
        self.after(self.cam_delay_ms, self._loop_video)

    def on_objeto_detectado(self, lbl_nombre, x, y, z):
        if self.limite_alcanzado: return
        
        if lbl_nombre not in self.historial_objetos:
            pts = 0
            if 'Negro' in lbl_nombre: pts = 5
            elif 'Blanco' in lbl_nombre: pts = 3
            elif 'Rojo' in lbl_nombre: pts = 1
            
            if self.puntos_actuales + pts > 10:
                self.historial_objetos.add(lbl_nombre)
                self.puntos_actuales += pts
                self.limite_alcanzado = True
                self.escaneando = False
                ctk.CTkLabel(self.lista_scroll, text=f"{lbl_nombre} (+{pts} pts) -> ¡LIMITE!").pack()
                self.lbl_puntos.configure(text=f'Puntos: {self.puntos_actuales} / 10 MAX', text_color='red')
                self.btn_escaneo.configure(text='Límite Alcanzado', state='disabled', fg_color='#7f8c8d')
                self.lbl_gp_status.configure(text=f'¡CANASTA LLENA! ({self.puntos_actuales} pts detectados)', text_color='red')
                return

            self.historial_objetos.add(lbl_nombre)
            self.puntos_actuales += pts
            ctk.CTkLabel(self.lista_scroll, text=f"{lbl_nombre} (+{pts} pts)").pack()
            self.lbl_puntos.configure(text=f'Puntos: {self.puntos_actuales} / 10 MAX')
            
            if self.puntos_actuales == 10:
                self.limite_alcanzado = True
                self.escaneando = False
                self.btn_escaneo.configure(text='Límite Alcanzado', state='disabled', fg_color='#7f8c8d')
                self.lbl_gp_status.configure(text=f'¡CANASTA LLENA! (10 pts)', text_color='red')
                self.lbl_puntos.configure(text_color='red')

    def _limpiar_todo(self):
        for w in self.lista_scroll.winfo_children(): w.destroy()
        self.historial_objetos.clear()
        self.puntos_actuales = 0
        self.limite_alcanzado = False
        self.lbl_puntos.configure(text='Puntos: 0 / 10 MAX', text_color='#3498db')
        self.lbl_gp_status.configure(text='')
        self.btn_escaneo.configure(state='normal', text='Iniciar Escaneo', fg_color='#1f6aa5')
        self.escaneando = False
        for c in self.temporizadores: self.temporizadores[c] = [0.0]*LIMITES_PELOTAS[c]
        for c in self.trackers: 
            for tr in self.trackers[c]: tr.reiniciar()

print('Celda 3 V12 lista.')

Celda 3 V12 lista.


In [ ]:
# Celda 4 - Punto de entrada  [V0.6]
if __name__ == '__main__':
    app = EscanerApp()
    app.mainloop()

Cargando modelo de IA (YOLOv8)...
